In [1]:
# parameters
datekey = 20260101

In [2]:
%%spark
--conf spark.jars.packages=com.microsoft.ml.spark:mmlspark_2.11:0.18.1
--conf spark.jars.ivySettings=/home/sankuai/.m2/ivysettings.xml
--conf spark.yarn.queue=root.zw03.hadoop-jiulvalgo.etl
--conf spark.executor.cores=4
--conf spark.task.cpus=4

,Driver日志,队列信息
,/home/sankuai/logs/spark-1783699984.log,root.zw03.hadoop-jiulvalgo.etl


CmdOutput(['tail', '-50', '/home/sankuai/logs/spark-1783699984.log'], 2)

:: loading settings :: file = /home/sankuai/.m2/ivysettings.xml


Ivy Default Cache set to: /home/sankuai/.ivy2/cache
The jars for the packages stored in: /home/sankuai/.ivy2/jars
:: loading settings :: url = jar:file:/opt/meituan/spark-3.0/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml
com.microsoft.ml.spark#mmlspark_2.11 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9639bf7d-cb05-4d84-a59b-2a4ef9e56a2c;1.0
	confs: [default]
	found com.microsoft.ml.spark#mmlspark_2.11;0.18.1 in mtdp
	found org.scalactic#scalactic_2.11;3.0.5 in mtdp
	found org.scala-lang#scala-reflect;2.11.12 in mtdp
	found org.scalatest#scalatest_2.11;3.0.5 in mtdp
	found org.scala-lang.modules#scala-xml_2.11;1.0.6 in mtdp
	found io.spray#spray-json_2.11;1.3.2 in mtdp
	found com.microsoft.cntk#cntk;2.4 in mtdp
	found org.openpnp#opencv;3.2.0-1 in mtdp
	found com.jcraft#jsch;0.1.54 in mtdp
	found org.apache.httpcomponents#httpclient;4.5.6 in mtdp
	found org.apache.httpcomponents#httpcore;4.4.10 in mtdp
	found commons-logging

,SparkSession,SparkContext,Job Search/applicationId
链接/变量名,spark,sc,application_1781180865561_1532666


In [3]:
%pip install lightgbm

Looking in indexes: https://pip.sankuai.com/simple/, https://pypi.sankuai.com/simple/
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install scikit-learn

Looking in indexes: https://pip.sankuai.com/simple/, https://pypi.sankuai.com/simple/
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install joblib

Looking in indexes: https://pip.sankuai.com/simple/, https://pypi.sankuai.com/simple/
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np
import json
import os, subprocess
import joblib

import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, roc_auc_score, classification_report, accuracy_score, precision_score, recall_score, f1_score

from pyspark.sql import functions as F
from pyspark.sql.functions import expr, when, col, udf, avg, lit, row_number
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

In [8]:
%%sql df --preview --quiet

select *
  from mart_jiulv_flow.dws_stp_session_comp_df
where dt = '{datekey}'

/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column_name] = series
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column_name] = series
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

,session_id,holiday_weekend_vacation_checkin_datekey_cnt,holiday_checkin_datekey_cnt,weekend_checkin_datekey_cnt,vacation_checkin_datekey_cnt,lead_time,checkin_datekey_cnt,goods_cnt,poi_cnt,aoi_cnt,city_cnt,stay_time,search_page_cnt,search_page_time,poi_list_cnt,poi_list_time,poi_info_cnt,poi_info_time,room_list_cnt,room_list_time,room_info_cnt,room_info_time,create_order_cnt,create_order_time,evaluate_page_cnt,evaluate_page_time,search_scenic_cnt,waitou_cnt,rednote_cnt,order_cnt,star2_cnt,star3_cnt,star4_cnt,star5_cnt,budget_cnt,express_cnt,business_cnt,theme_cnt,couple_cnt,apartment_cnt,inn_cnt,homestay_cnt,hostel_cnt,farmstay_cnt,family_guesthouse_cnt,guesthouse_cnt,resort_hotel_cnt,villa_cnt,family_cnt,esports_cnt,scarce_cnt,unique_cnt,lowprice_cnt,good_cnt,tourism_core_city_cnt,tourism_big_city_cnt,tourism_seasonal_city_cnt,tier1_city_cnt,new_tier1_city_cnt,tier2_city_cnt,tier3_city_cnt,tier4_city_cnt,tier5_city_cnt,university_cnt,transportation_hub_cnt,scenic_area_cnt,hospital_cnt,performance_sports_venue_cnt,convention_center_cnt,industrial_park_cnt,king_room_cnt,single_room_cnt,double_room_cnt,triple_room_cnt,suite_cnt,standalone_cnt,dorm_cnt,is_college_student,is_adult_single,is_adult_married_no_kids,is_adult_married_with_kids,is_over_60,is_senior,is_middle_age,is_young,is_minor,is_high_value,is_mid_value,is_low_value,is_l4,is_l1_3,is_youth_campus,is_business_elite,is_self_social,is_practical_life,is_family_guardian,is_vital_soldier,is_quality_visitor,is_kid_explorer,is_occasional_traveler,holiday_rnt_pct,travel_distance,is_holiday_weekend_vacation,is_holiday,is_weekend,is_vacation,order_lead_time,is_star2,is_star3,is_star4,is_star5,is_budget,is_express,is_business,is_theme,is_couple,is_apartment,is_inn,is_homestay,is_hostel,is_farmstay,is_family_guesthouse,is_guesthouse,is_resort_hotel,is_villa,is_family,is_esports,is_scarce,is_unique,is_lowprice,is_good,order_travel_distance,is_search_scenic,is_waitou,is_rednote,is_tourism_core_city,is_tourism_big_city,is_tourism_seasonal_city,is_tier1_city,is_new_tier1_city,is_tier2_city,is_tier3_city,is_tier4_city,is_tier5_city,is_university,is_transportation_hub,is_scenic_area,is_hospital,is_performance_sports_venue,is_convention_center,is_industrial_park,is_king_room,is_single_room,is_double_room,is_triple_room,is_suite,is_standalone,is_dorm,search_cnt,travel_session_cnt,is_search,is_travel_session,session_cnt,event_cnt,advance_checkin_datekey_cnt,sameday_checkin_datekey_cnt,overnight_checkin_datekey_cnt,dt,level3_biz_code
0,78da60c2-8091-44aa-ba41-2b7843a689a81783564304070520,13,0,8,9,4.711514,22,30,34,1,1,15045.902,0,0.000,0,0.000,632,7595.043,0,0.0,0,0.0,3,72.417,0,0.0,0,0,0,0,833,21,0,0,684,10,0,0,0,31,0,94,0,14,18,0,0,0,22,0,0,1,113,114,0,854,0,0,0,854,0,0,0,0,0,854,0,0,0,0,59,5,85,14,22,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,1,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.561507,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,41,871,21,6,1,20260709,hotel
1,AE6116F6-4DE6-4811-A02C-F099BA07D8971783560286049254,5,0,4,1,1.338462,11,10,6,1,1,1469.353,0,0.000,0,0.000,250,1366.804,0,0.0,0,0.0,0,0.000,0,0.0,0,0,0,0,263,3,0,0,204,0,0,0,0,0,0,60,0,0,0,0,0,0,207,0,0,238,28,266,0,266,0,0,266,0,0,0,0,0,0,266,0,0,0,0,18,0,8,1,1,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,27.232850,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,32,266,10,6,1,20260709,hotel
2,2C448B5A-51C0-46FF-9222-5DC04D4772201783530182315287,0,0,0,0,-0.818182,1,4,11,4,1,794.083,2,4.945,12,283.615,20,248.063,0,0.0,0,0.0,0,0.000,0,0.0,0,9,0,0,14,11,0,0,19,0,6,0,0,0,0,0,0,0,0,0,0,0,2,0,0,7,18,25,25,0,0,0,0,0,25,0,0,0,19,5,0,1,0,0,7,1,2,0,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.055249,4.019904,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,

In [9]:
def metrics(df):
    # time
    df = df.withColumn('workday_cnt',col('checkin_datekey_cnt')-col('holiday_weekend_vacation_checkin_datekey_cnt'))
    df = df.withColumn('is_workday',1-col('is_holiday_weekend_vacation'))
    df = df.withColumn("is_advance_booking",when(col("order_lead_time") > 0, 1).otherwise(0))
    df = df.withColumn("is_sameday_booking",when(col("order_lead_time") == 0, 1).otherwise(0))
    df = df.withColumn("is_overnight_booking",when(col("order_lead_time") < 0, 1).otherwise(0))
    df = df.withColumn("is_holiday_weekend_vacation_advance_booking",when((col('holiday_weekend_vacation_checkin_datekey_cnt')>0)&(col('advance_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_holiday_weekend_vacation_sameday_booking",when((col('holiday_weekend_vacation_checkin_datekey_cnt')>0)&(col('sameday_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_holiday_weekend_vacation_overnight_booking",when((col('holiday_weekend_vacation_checkin_datekey_cnt')>0)&(col('overnight_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_workday_advance_booking",when((col('workday_cnt')>0)&(col('advance_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_workday_sameday_booking",when((col('workday_cnt')>0)&(col('sameday_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_workday_overnight_booking",when((col('workday_cnt')>0)&(col('overnight_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_order_holiday_weekend_vacation_advance_booking",when((col('is_holiday_weekend_vacation')==1)&(col('is_advance_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_holiday_weekend_vacation_sameday_booking",when((col('is_holiday_weekend_vacation')==1)&(col('is_sameday_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_holiday_weekend_vacation_overnight_booking",when((col('is_holiday_weekend_vacation')==1)&(col('is_overnight_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_workday_advance_booking",when((col('is_workday')==1)&(col('is_advance_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_workday_sameday_booking",when((col('is_workday')==1)&(col('is_sameday_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_workday_overnight_booking",when((col('is_workday')==1)&(col('is_overnight_booking')==1),1).otherwise(0))
    
    # distance
    df = df.withColumn("is_long_distance",when(col('travel_distance')>1000,1).otherwise(0))
    df = df.withColumn("is_mid_distance",when((col('travel_distance')>301)&(col('travel_distance')<=1000),1).otherwise(0))
    df = df.withColumn("is_nearby_distance",when((col('travel_distance')>51)&(col('travel_distance')<=300),1).otherwise(0))
    df = df.withColumn("is_short_distance",when(col('travel_distance')<=50,1).otherwise(0))
    df = df.withColumn("is_order_long_distance",when(col('order_travel_distance')>1000,1).otherwise(0))
    df = df.withColumn("is_order_mid_distance",when((col('order_travel_distance')>301)&(col('order_travel_distance')<=1000),1).otherwise(0))
    df = df.withColumn("is_order_nearby_distance",when((col('order_travel_distance')>51)&(col('order_travel_distance')<=300),1).otherwise(0))
    df = df.withColumn("is_order_short_distance",when(col('order_travel_distance')<=50,1).otherwise(0))
    
    # city
    df = df.withColumn('high_tier_city_cnt',col('tier1_city_cnt') +col('new_tier1_city_cnt') +col('tier2_city_cnt'))
    df = df.withColumn('low_tier_city_cnt',col('tier3_city_cnt') +col('tier4_city_cnt') +col('tier5_city_cnt'))
    df = df.withColumn('high_tier_city_ratio',when(col('city_cnt') > 0,col('high_tier_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('low_tier_city_ratio',when(col('city_cnt') > 0,col('low_tier_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('is_high_tier_city',when((col('is_tier1_city')==1)|(col('is_new_tier1_city')==1)|(col('is_tier2_city')==1),1).otherwise(0))
    df = df.withColumn('is_low_tier_city',when((col('is_tier3_city')==1)|(col('is_tier4_city')==1)|(col('is_tier5_city')==1),1).otherwise(0))
    df = df.withColumn('tourism_core_city_ratio',when(col('city_cnt') > 0,col('tourism_core_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('tourism_big_city_ratio',when(col('city_cnt') > 0,col('tourism_big_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('tourism_seasonal_city_ratio',when(col('city_cnt') > 0,col('tourism_seasonal_city_cnt') / col('city_cnt')).otherwise(0))
    
    # aoi
    df = df.withColumn('university_ratio', when(col('aoi_cnt')>0, col('university_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('transportation_hub_ratio', when(col('aoi_cnt')>0, col('transportation_hub_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('scenic_area_ratio', when(col('aoi_cnt')>0, col('scenic_area_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('hospital_ratio', when(col('aoi_cnt')>0, col('hospital_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('performance_sports_venue_ratio', when(col('aoi_cnt')>0, col('performance_sports_venue_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('convention_center_ratio', when(col('aoi_cnt')>0, col('convention_center_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('industrial_park_ratio', when(col('aoi_cnt')>0, col('industrial_park_cnt') / col('aoi_cnt')).otherwise(0))
    
    # poi
    df = df.withColumn('high_star_cnt',col('star3_cnt') +col('star4_cnt') +col('star5_cnt'))
    df = df.withColumn('is_high_star',when((col('is_star3')==1)|(col('is_star4')==1)|(col('is_star5')==1),1).otherwise(0))
    df = df.withColumn('high_star_ratio',when(col('poi_cnt') > 0,col('high_star_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('low_star_ratio',when(col('poi_cnt') > 0,col('star3_cnt') / col('poi_cnt')).otherwise(0))
    
    df = df.withColumn('budget_ratio', when(col('poi_cnt')>0, col('budget_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('express_ratio', when(col('poi_cnt')>0, col('express_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('business_ratio', when(col('poi_cnt')>0, col('business_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('theme_ratio', when(col('poi_cnt')>0, col('theme_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('couple_ratio', when(col('poi_cnt')>0, col('couple_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('apartment_ratio', when(col('poi_cnt')>0, col('apartment_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('inn_ratio', when(col('poi_cnt')>0, col('inn_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('homestay_ratio', when(col('poi_cnt')>0, col('homestay_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('hostel_ratio', when(col('poi_cnt')>0, col('hostel_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('farmstay_ratio', when(col('poi_cnt')>0, col('farmstay_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('family_guesthouse_ratio', when(col('poi_cnt')>0, col('family_guesthouse_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('guesthouse_ratio', when(col('poi_cnt')>0, col('guesthouse_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('resort_hotel_ratio', when(col('poi_cnt')>0, col('resort_hotel_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('villa_ratio', when(col('poi_cnt')>0, col('villa_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('family_ratio', when(col('poi_cnt')>0, col('family_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('esports_ratio', when(col('poi_cnt')>0, col('esports_cnt') / col('poi_cnt')).otherwise(0))
    
    df = df.withColumn('scarce_ratio', when(col('poi_cnt')>0, col('scarce_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('unique_ratio', when(col('poi_cnt')>0, col('unique_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('lowprice_ratio', when(col('poi_cnt')>0, col('lowprice_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('good_ratio', when(col('poi_cnt')>0, col('good_cnt') / col('poi_cnt')).otherwise(0))
    
    # goods
    df = df.withColumn('king_room_ratio', when(col('goods_cnt')>0, col('king_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('single_room_ratio', when(col('goods_cnt')>0, col('single_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('double_room_ratio', when(col('goods_cnt')>0, col('double_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('triple_room_ratio', when(col('goods_cnt')>0, col('triple_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('suite_ratio', when(col('goods_cnt')>0, col('suite_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('standalone_ratio', when(col('goods_cnt')>0, col('standalone_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('dorm_ratio', when(col('goods_cnt')>0, col('dorm_cnt') / col('goods_cnt')).otherwise(0))
    
    # browsing
    df = df.withColumn('avg_checkin_datekey_cnt',when(col('session_cnt')>0,col('checkin_datekey_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_city_cnt',when(col('session_cnt')>0,col('city_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_aoi_cnt',when(col('session_cnt')>0,col('aoi_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_poi_cnt',when(col('session_cnt')>0,col('poi_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_goods_cnt',when(col('session_cnt')>0,col('goods_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_stay_time',when(col('session_cnt')>0,col('stay_time')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_order_cnt',when(col('session_cnt')>0,col('order_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_search_cnt',when(col('session_cnt')>0,col('search_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_travel_session_cnt',when(col('session_cnt')>0,col('travel_session_cnt')/col('session_cnt')).otherwise(0))
    
    df = df.withColumn('search_page_time_ratio',when(col('stay_time')>0,col('search_page_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('poi_list_time_ratio',when(col('stay_time')>0,col('poi_list_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('poi_info_time_ratio',when(col('stay_time')>0,col('poi_info_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('room_list_time_ratio',when(col('stay_time')>0,col('room_list_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('room_info_time_ratio',when(col('stay_time')>0,col('room_info_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('create_order_time_ratio',when(col('stay_time')>0,col('create_order_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('evaluate_page_time_ratio',when(col('stay_time')>0,col('evaluate_page_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('search_page_cnt_ratio',when(col('event_cnt')>0,col('search_page_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('poi_list_cnt_ratio',when(col('event_cnt')>0,col('poi_list_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('poi_info_cnt_ratio',when(col('event_cnt')>0,col('poi_info_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('room_list_cnt_ratio',when(col('event_cnt')>0,col('room_list_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('room_info_cnt_ratio',when(col('event_cnt')>0,col('room_info_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('create_order_cnt_ratio',when(col('event_cnt')>0,col('create_order_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('evaluate_page_cnt_ratio',when(col('event_cnt')>0,col('evaluate_page_cnt')/col('event_cnt')).otherwise(0))
    
    df = df.withColumn('search_non_scenic_cnt',col('search_cnt')-col('search_scenic_cnt'))
    df = df.withColumn('search_scenic_ratio',when(col('search_cnt')>0,col('search_scenic_cnt')/col('search_cnt')).otherwise(0))
    df = df.withColumn('search_non_scenic_ratio',when(col('search_cnt')>0,col('search_non_scenic_cnt')/col('search_cnt')).otherwise(0))
    
    return df

In [10]:
df = metrics(df)

In [8]:
%%sql rela --preview --quiet

select session_id,related_session_id
from mart_jiulv_flow.dim_stp_session_rel_df
where dt = '{datekey}'

,session_id,related_session_id
0,5E290D25-D784-47AD-AA22-5AF822D6EF891782865654541404,5E290D25-D784-47AD-AA22-5AF822D6EF891782865654541404
1,c212baa9-d97e-41fa-a70f-89e692cab08c1782915058400406,c212baa9-d97e-41fa-a70f-89e692cab08c1782638338936752
2,664F09CC-0E32-4C9E-9902-C2AB71B692E51782920623910316,664F09CC-0E32-4C9E-9902-C2AB71B692E51782491042879973
3,948564b8-28a1-4deb-9be7-616b6bd7ce701782905080272654,948564b8-28a1-4deb-9be7-616b6bd7ce701782704000344467
4,9017a090-9277-461b-80b7-a92a48fa26a4178290025686614,e0318c9f-913f-4996-9503-c5b27cf591511782699941537723
5,aa79effb-ebb5-4d23-ad27-fa65ece065921782915821811103,aa79effb-ebb5-4d23-ad27-fa65ece065921782653351796904
6,0930b186-9cc1-410d-a633-cb212a52ed011782865874920485,0930b186-9cc1-410d-a633-cb212a52ed011782865874920485
7,D84A09A1-B2F1-496D-98E0-2C2808751D331782889168855635,D84A09A1-B2F1-496D-98E0-2C2808751D331782889168855635
8,D84A09A1-B2F1-496D-98E0-2C2808751D331782861440642882,D84A09A1-B2F1-496D-98E0-2C2808751D331782683360667815
9,C6FFB73E-9BDD-4273-AE59-AF10BBEA52531782876225125509,C6FFB73E-9BDD-4273-AE59-AF10BBEA52531782876225125509


In [11]:
model_hdfs = "/user/hadoop-jiulvsearch-algo/models/lgb_intention_model_v2.txt"
meta_hdfs = "/user/hadoop-jiulvsearch-algo/models/lgb_intention_model_v2.txt.meta"
iso_hdfs = "/user/hadoop-jiulvsearch-algo/models/lgb_intention_model_v2_iso.pkl"
model_local = "/tmp/lgb_intention_model_v2.txt"
meta_local = "/tmp/lgb_intention_model_v2.txt.meta"
iso_local = "/tmp/lgb_intention_model_v2_iso.pkl"

for hdfs_path, local_path in [(model_hdfs, model_local), (meta_hdfs, meta_local), (iso_hdfs, iso_local)]:
    if os.path.exists(local_path):
        os.remove(local_path)
    subprocess.run(["hdfs", "dfs", "-get", hdfs_path, local_path], check=True)

with open(meta_local, 'r') as f:
    best_threshold = json.load(f)['best_threshold']

bst = lgb.Booster(model_file=model_local)
iso = joblib.load(iso_local)

In [12]:
total = df.count()
n_buckets = max(1, ((total + 799999) // 800000))
print("n_bucket:",n_buckets)
df = df.repartition(n_buckets)
df = df.withColumn("bucket",F.spark_partition_id())

In [ ]:
df.cache()
df.count()

In [13]:
def predict_batch(sdf):
    pdf = sdf.toPandas()
    X = pdf.drop(columns=['intention','session_id','dt','level3_biz_code'], errors='ignore')
    X = X.reindex(columns=bst.feature_name(), fill_value=0)
    y_pred_prob_raw = bst.predict(X)
    y_pred_prob = iso.transform(y_pred_prob_raw)
    y_pred = (y_pred_prob >= best_threshold).astype(int)
    result = pdf[['session_id','dt']].copy()
    result['intention'] = y_pred
    result['intention_prob'] = y_pred_prob
    return spark.createDataFrame(
        result.values.tolist(),
        schema="session_id string, dt string, intention int, intention_prob double"
    )

In [14]:
results = []
for i in range(n_buckets):
    sdf_i = df.filter(F.col("bucket") == i)
    if sdf_i.rdd.isEmpty():
        continue
    print(f"[bucket {i}] start predict")
    result_i = predict_batch(sdf_i)
    print(f"[bucket {i}] done, rows={result_i.count()}")
    results.append(result_i)

[bucket 1] start predict


KeyboardInterrupt: 

In [ ]:
from functools import reduce
results = reduce(lambda x, y: x.unionByName(y), results)
output = rela.alias("a").join(results.alias("b"),F.col("a.session_id") == F.col("b.session_id"),"inner").select(F.col("a.related_session_id").alias("session_id"),F.col("b.intention"),F.col("b.intention_prob"),F.col("b.dt"))
output = output.groupBy("session_id", "dt").agg(avg("intention_prob").alias("intention_prob"))
output = output.withColumn("intention",when(col("intention_prob") >= best_threshold, 1).otherwise(0))
outputs = output.createOrReplaceTempView("outputs")

In [ ]:
%%sql r --preview --quiet

SET hive.exec.dynamic.partition=true

In [ ]:
%%sql r --preview --quiet

SET hive.exec.dynamic.partition.mode=nonstrict

In [ ]:
%%sql r --preview --quiet

INSERT INTO TABLE mart_jiulvsearch_algo.stp_result PARTITION(dt)
SELECT 
    session_id,
    intention,
    intention_prob,
    dt
FROM outputs